# Initial Datahandeling

### This notebook shows the initial datahandling of utterences in the danihs Parliament.
### It includes the following:

- Loading the two overlapping datasets of utterences in the danish parliament
    - Due to the ParlLawSpeech data being in .rds format, it was read into r, converted to .csv format and then saved in this repo
- Merging them by end data of corp_v2
- Sentence segmentation
- Reformatting to json object for easier handling during labeling for validation, training and inference
- Datacleaning
- Splitting into training data (including validationset to be extracted), and inference data

In [3]:
# Load packages

import pandas as pd
import os


# Corp_Folketing_V2

In [4]:
#read the corp folketing 

corp_v2_path = os.path.join("..",
                            "..",
                            "raw_data",
                            "Corp_Folketing_V2.csv")

# Read the two datasets
df_corp = pd.read_csv(corp_v2_path)
x = df_corp.pop("Unnamed: 0") #dummy variable

#make date a datetime object
df_corp["date"] = pd.to_datetime(df_corp["date"])

In [5]:
#This dataset spans the following time period

min_time_corp = df_corp["date"].min()
max_time_corp = df_corp["date"].max()

print(f"The Corp dataset has coverage from {min_time_corp.date()} to {max_time_corp.date()}")

The Corp dataset has coverage from 1997-10-07 to 2018-12-20


# PLS speeches

In [6]:
#read the PLS data. 
# OBS has been made to .csv in file ./PLS_rds_to_csv.RMD

PLS_path = os.path.join("..",
                            "..",
                            "..",
                            "..",
                            "Data_outside_git",
                            "PLS_speeches.csv")

# Read the two datasets
df_PLS = pd.read_csv(PLS_path)

#make date a datetime object
df_PLS["date"] = pd.to_datetime(df_PLS["date"])

In [7]:
#This dataset spans the following time period

min_time_PLS = df_PLS["date"].min()
max_time_PLS = df_PLS["date"].max()

print(f"The PLS dataset has coverage from {min_time_PLS.date()} to {max_time_PLS.date()}")

The PLS dataset has coverage from 2007-11-27 to 2022-10-06


# Merging datasets

In [8]:

# Keep only rows in df2 strictly after max date of df1
df_PLS_filtered = df_PLS[df_PLS['date'] > max_time_corp]

# Concatenate
df_merged = pd.concat([df_corp, df_PLS_filtered], ignore_index=True)

# sort by date
df_merged = df_merged.sort_values('date').reset_index(drop=True)

In [9]:
#check that the span now is the entire period
#This dataset spans the following time period

min_time_merged = df_merged["date"].min()
max_time_merged = df_merged["date"].max()

print(f"The merged datasets has coverage from {min_time_merged.date()} to {max_time_merged.date()}")

The merged datasets has coverage from 1997-10-07 to 2022-10-06


# Now for sentence segmentation?

Implementatino of the DaCy pipeline. Due to incompatibility with newer python versions, python version 3.10.19 used. Spacy version 3.5.4 has been used, as the model card on huggingface for da_dacy_large_trf specifies spacy to be >=3.5.2,<3.6.0.


*Personal note: use env: dacy_env*


In [83]:
#import packages for sentence segmentation

import spacy
import dacy
import os

import json
from pathlib import Path
from tqdm import tqdm

In [79]:
for model in dacy.models():
    print(model)

da_dacy_small_trf-0.2.0
da_dacy_medium_trf-0.2.0
da_dacy_large_trf-0.2.0
small
medium
large
da_dacy_small_ner_fine_grained-0.1.0
da_dacy_medium_ner_fine_grained-0.1.0
da_dacy_large_ner_fine_grained-0.1.0


## Issues

While this downloading process should be straight forward with nlp = spacy.load("da_dacy_large_trf-0.2.0"), this does not work.
Thus, the da_dacy_large_trf-any-py3-none-any.whl file has been manually from huggingface, and the path specified
- dacy_path = os.path.abspath("C:/Users/marku/Desktop/AU/Bachelor/Data_outside_git/da_dacy_large_trf-any-py3-none-any.whl")

The file must be renamed for the pipeline to accept it
- #rename da_dacy_large_trf-any-py3-none-any.whl da_dacy_large_trf-0.2.0-py3-none-any.whl

And then manually installed
- python -m pip install da_dacy_large_trf-0.2.0-py3-none-any.whl --no-deps

Verification that spacy sees the model
- import spacy
- print(spacy.util.get_installed_models())

However, dacy still does not recognize, and since it has been build on spacy infrastructure, spacy is used for loading the model instead
- nlp = spacy.load("da_dacy_large_trf")

The name of the model can be double chekced
- print(nlp.meta["name"]) # shoudl say the above name

In [80]:
nlp = spacy.load("da_dacy_large_trf") #dacy load does not work, but spacy with the dacy corpus
print(nlp.meta["name"]) # shoudl say the above name

c:\Users\marku\miniconda3\envs\dacy_env\lib\site-packages\spacy_transformers\layers\hf_shim.py:137: UserWarning: Error loading saved torch state_dict with strict=True, likely due to differences between 'transformers' versions. Attempting to load with strict=False as a fallback...

If you see errors or degraded performance, download a newer compatible model or retrain your custom model with the current 'transformers' and 'spacy-transformers' versions. For more details and available updates, run: python -m spacy validate
  warnings.warn(warn_msg)


dacy_large_trf


In [81]:
#Setup example
doc = nlp(
    "Jeg skal blot beklage, at hr. Poul Erik Dyrlund tilsyneladende ikke forstår sit eget lovforslag. Noget efterfølgende sætninger"
)

c:\Users\marku\miniconda3\envs\dacy_env\lib\site-packages\thinc\shims\pytorch.py:114: FutureWarning: `torch.cuda.amp.autocast(args...)` is deprecated. Please use `torch.amp.autocast('cuda', args...)` instead.
  with torch.cuda.amp.autocast(self._mixed_precision):


In [84]:
for sent in doc.sents:
    print(sent)

Jeg skal blot beklage, at hr. Poul Erik Dyrlund tilsyneladende ikke forstår sit eget lovforslag.
Noget efterfølgende sætninger


In [ ]:
# lopp trough all rows of the dataset
# Do sentence segmentation and reformat to json

#works but inefficient

output_path = Path(os.path.join("..",
                           "..",
                           "raw_data",
                           "test_sentences.jsonl"))

all_sentences = []


with output_path.open("w", encoding="utf-8") as f:

    for paragraph, text in tqdm(enumerate(df_merged["text"])):
        
        doc = nlp(text)
        row = df_merged.iloc[paragraph]

        for i, sent in enumerate(doc.sents):
            
            sentence_data = {"date": row["date"].strftime("%m/%d/%Y"), 
            "speaker": row["speaker"], 
            "party": row["party"],
            "paragraph_nr" : paragraph, 
            "sentence_nr": i, 
            "text": str(sent)}
            
            #all_sentences.append(sentence_data)
            f.write(json.dumps(sentence_data, ensure_ascii=False) + "\n")